In [2]:
!pip uninstall google-cloud-pipeline-components -y
!pip install --no-cache-dir google-cloud-pipeline-components

Found existing installation: google-cloud-pipeline-components 2.17.0
Uninstalling google-cloud-pipeline-components-2.17.0:
  Successfully uninstalled google-cloud-pipeline-components-2.17.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 90.4 MB/s eta 0:00:00


In [6]:
!pip install google-cloud-aiplatform kfp jinja2 google-api-core

In [9]:
!pip show google-cloud-pipeline-components

Name: google-cloud-pipeline-components
Version: 2.17.0
Summary: This SDK enables a set of First Party (Google owned) pipeline components that allow users to take their experience from Vertex AI SDK and other Google Cloud services and create a corresponding pipeline using KFP or Managed Pipelines.
Home-page: https://github.com/kubeflow/pipelines/tree/master/components/google-cloud
Author: The Google Cloud Pipeline Components authors
Author-email: google-cloud-pipeline-components@google.com
License: Apache License 2.0
Location: /opt/conda/lib/python3.10/site-packages
Requires: google-api-core, google-cloud-aiplatform, Jinja2, kfp
Required-by: 


In [1]:
!pip show google-api-core google-cloud-aiplatform kfp

Name: google-api-core
Version: 2.20.0
Summary: Google API client core library
Home-page: https://github.com/googleapis/python-api-core
Author: Google LLC
Author-email: googleapis-packages@google.com
License: Apache 2.0
Location: /home/jupyter/.local/lib/python3.10/site-packages
Requires: google-auth, googleapis-common-protos, proto-plus, protobuf, requests
Required-by: google-api-python-client, google-cloud-aiplatform, google-cloud-artifact-registry, google-cloud-bigquery, google-cloud-bigquery-connection, google-cloud-bigquery-storage, google-cloud-core, google-cloud-dataproc, google-cloud-datastore, google-cloud-functions, google-cloud-iam, google-cloud-language, google-cloud-monitoring, google-cloud-pipeline-components, google-cloud-resource-manager, google-cloud-storage, kfp, opencensus
---
Name: google-cloud-aiplatform
Version: 1.67.1
Summary: Vertex AI API client library
Home-page: https://github.com/googleapis/python-aiplatform
Author: Google LLC
Author-email: googleapis-package

In [12]:
!pip install -U google-cloud-pipeline-components

In [1]:
from google_cloud_pipeline_components.v1.dataproc import DataprocPySparkBatchOp
from kfp.v2 import dsl
from kfp.v2 import compiler

import kfp
from google.cloud import aiplatform

/var/tmp/ipykernel_9039/1328720654.py:2: DeprecationWarning: The module `kfp.v2` is deprecated and will be removed in a futureversion. Please import directly from the `kfp` namespace, instead of `kfp.v2`.
  from kfp.v2 import dsl


In [26]:
PROJECT_ID = 'capable-hash-432501-a6'
REGION = 'us-east4'
BUCKET_NAME = 'worshop-vertex-pip'
PYSPARK_URI = f'gs://{BUCKET_NAME}/src/model_training.py'
MODEL_URI = f'gs://{BUCKET_NAME}/model'
METRICS_URI = f'gs://{BUCKET_NAME}/metrics/metrics.json'
CONTAINER_IMAGE = 'gcr.io/spark-operator/spark-py:v3.1.1'

SERVICE_ACCOUNT = '815942598901-compute@developer.gserviceaccount.com'

In [32]:
@dsl.pipeline(
    name="iris-pyspark-pipeline",
    description="Pipeline to train a Decision Tree model on the Iris dataset using PySpark and Dataproc Serverless."
)
def pipeline():
    # Define the PySpark task
    pyspark_task = DataprocPySparkBatchOp(
        project=PROJECT_ID,
        location=REGION,
        container_image=CONTAINER_IMAGE,
        batch_id="pysparktrainjob",
        main_python_file_uri=PYSPARK_URI,
        args=[
            '--model_path', MODEL_URI,
            '--metrics_path', METRICS_URI,
        ]
    )

In [33]:
# Compile the pipeline
compiler.Compiler().compile(
    pipeline_func=pipeline, 
    package_path='iris_pipeline.json')

In [34]:
# Inicializa Vertex AI en la región deseada
aiplatform.init(
    project=PROJECT_ID,
    location=REGION,  # Asegúrate de definir la región aquí
)

In [35]:
# Submit the pipeline job
pipeline_job = aiplatform.PipelineJob(
    display_name="iris-pyspark-pipeline",
    template_path="iris_pipeline.json",
    pipeline_root=f'gs://{BUCKET_NAME}/pipeline_root'
)

In [36]:
# Inicia el pipeline
pipeline_job.run(service_account=SERVICE_ACCOUNT)

Creating PipelineJob
PipelineJob created. Resource name: projects/815942598901/locations/us-east4/pipelineJobs/iris-pyspark-pipeline-20241008065319
To use this PipelineJob in another session:
pipeline_job = aiplatform.PipelineJob.get('projects/815942598901/locations/us-east4/pipelineJobs/iris-pyspark-pipeline-20241008065319')
View Pipeline Job:
https://console.cloud.google.com/vertex-ai/locations/us-east4/pipelines/runs/iris-pyspark-pipeline-20241008065319?project=815942598901
PipelineJob projects/815942598901/locations/us-east4/pipelineJobs/iris-pyspark-pipeline-20241008065319 current state:
PipelineState.PIPELINE_STATE_RUNNING
PipelineJob projects/815942598901/locations/us-east4/pipelineJobs/iris-pyspark-pipeline-20241008065319 current state:
PipelineState.PIPELINE_STATE_RUNNING


RuntimeError: Job failed with:
code: 9
message: " The DAG failed because some tasks failed. The failed tasks are: [dataproc-create-pyspark-batch].; Job (project_id = capable-hash-432501-a6, job_id = 6057537211883061248) is failed due to the above error.; Failed to handle the job: {project_number = 815942598901, job_id = 6057537211883061248}"
